# TotalSegmentator inference (nb2): converted NIfTI → segmentations  —  model-specific

Runs TotalSegmentator (v2.18.0) on the GPU VM. Consumes the
**Boundary-A** archive `converted_nifti.tar.lz4` from nb1 and emits the **Boundary-B**
archive `segmentations.tar.lz4` with the canonical layout (one `<task>` dir per
requested task — the same multi-model layout MOOSE emits, which nb3 already handles):
```
<SeriesInstanceUID>/<task>/segmentations/<SeriesInstanceUID>.nii.gz
<SeriesInstanceUID>/<task>/label_map.json      # {label_id: label_name}
```

The `task` papermill parameter is a comma/space-separated list (mirroring MOOSE's
`moose_models`), e.g. `task: total,lung_vessels` to run both on one VM and share
the download/convert/boot cost. Supported tasks:

- **`total`** (default): 117-structure full-body segmentation (`--ml` multilabel;
  v2 merged the v1 heart chambers into a single `heart` and added 20 new
  structures — sternum, spinal_cord, thyroid_gland, costal_cartilages, ...).
  Label IDs from `class_map['total']`.
- **`lung_vessels`**: task 117, retrained in v2 as 4 classes (`lung_airways`,
  `lung_airways_wall`, `lung_arteries`, `lung_veins`). v2 crops the FOV to the
  lung lobes internally (using the baked 3 mm model), so this is a single call —
  the v1 two-step pre-segmentation pipeline is gone. `fast=True` applies only to
  the `total` task (ignored for subtasks in a mixed list).
  Label IDs from `class_map['lung_vessels']`.

With `--ml`, v2 treats `-o` as the output **file** path (v1 treated it as a
directory). Weights for both tasks (plus the 3 mm crop model) are baked into the
image under `TOTALSEG_HOME_DIR`; nothing is fetched at job runtime. Checkpointing
is per (series, task), so a preempted VM resumes mid-task-list.

nb3 (shared) turns the output into DICOM-SEG + radiomics + SR using the
SNOMED mapping rows that match each task's label names.

## Imports

In [ ]:
import json
import shutil
import subprocess
import time
import traceback
from pathlib import Path

NOTEBOOK_START = time.time()
def _elapsed(s=None):
    return f"{time.time() - (s if s is not None else NOTEBOOK_START):.1f}s"
print(f"[T+{_elapsed()}] Imports complete")

## Parameters

In [ ]:
# Boundary-A archive produced by nb1 (local file on the same VM).
converted_nifti_path = "converted_nifti.tar.lz4"

# Workflow-level model tag (used in metrics; each task gets its own
# <uid>/<task>/ dir in the Boundary-B layout, like MOOSE's sub-models).
model_name = "total"

# 'cuda' for GPU, 'cpu' for CPU-only.
accelerator = "cuda"

# Optional GCS prefix (gs://bucket/prefix) for checkpoint/resume on preemption: each
# finished (series, task) output is saved there and a retried VM skips it. run_id (the
# Cromwell workflow id, from the WDL) namespaces the checkpoint. Empty = disabled.
checkpoint_gcs = ""
run_id = ""

# Model-specific knobs injected via `papermill -f inference_params.yaml`.
# fast=True uses TotalSegmentator's 3mm model (faster, lower resolution);
# only valid for the 'total' task.
fast = False

# TotalSegmentator task(s), comma/space-separated — mirrors MOOSE's moose_models:
#   'total'                    (default, 117 structures in v2)
#   'lung_vessels'             (4 classes in v2: airways, airway walls, arteries, veins)
#   'total,lung_vessels'       (both on one VM: shared download/convert/boot)
task = "total"

## Extract Boundary-A archive

In [ ]:
NIFTI_DIR = Path('/tmp/converted_nifti')
SEG_DIR = Path('/tmp/segmentations')
for _d in (NIFTI_DIR, SEG_DIR):
    if _d.exists():
        shutil.rmtree(_d)
    _d.mkdir(parents=True, exist_ok=True)

subprocess.run(f'lz4 -d -c {converted_nifti_path} | tar -xf - -C {NIFTI_DIR.parent}',
               shell=True, check=True)
if (NIFTI_DIR / 'converted_nifti').is_dir():
    NIFTI_DIR = NIFTI_DIR / 'converted_nifti'
series_uids = sorted(d.name for d in NIFTI_DIR.iterdir() if d.is_dir())
print(f'Series : {len(series_uids)}  |  task={task}  fast={fast}')

# ---- checkpoint / resume (segmentator_checkpoint.py is fetched next to the notebook by
#      the WDL; no-op when checkpoint_gcs is empty) ----
import os, sys
for _p in (os.getcwd(), str(Path.cwd())):
    if _p not in sys.path:
        sys.path.insert(0, _p)
try:
    from segmentator_checkpoint import Checkpointer
except ImportError:
    Checkpointer = None
if checkpoint_gcs and Checkpointer is None:
    raise RuntimeError('checkpoint_gcs is set but segmentator_checkpoint.py was not found '
                       'next to the notebook (the WDL fetches it from gitRepo/gitBranch)')
ckpt = Checkpointer(checkpoint_gcs, run_id) if Checkpointer else None
completed_seg = ckpt.restore_segs(SEG_DIR) if ckpt else set()
if completed_seg:
    print(f'[T+{_elapsed()}] {len(completed_seg)} (series, task) outputs restored from checkpoint')

## Authoritative label map from TotalSegmentator's class map

In [ ]:
import re

from totalsegmentator.map_to_binary import class_map

VALID_TASKS = ('total', 'lung_vessels')
tasks = [t for t in re.split(r'[,\s;]+', str(task).strip()) if t]
unknown = [t for t in tasks if t not in VALID_TASKS]
if unknown or not tasks:
    raise ValueError(f"Unknown task(s) {unknown or task!r}; valid tasks: {VALID_TASKS}")

TASK_LABELS = {t: {str(k): v for k, v in class_map[t].items()} for t in tasks}
for t in tasks:
    print(f'Loaded {len(TASK_LABELS[t])} TotalSegmentator label ids for task={t}')

## Run TotalSegmentator → Boundary-B layout

In [ ]:
errors = []
usage_metrics = {'series': {}}

def _run_task(nii, work, ts_task, ts_fast):
    """One TotalSegmentator v2 call producing a multilabel volume. With --ml, -o is
    the output FILE path. Subtasks (lung_vessels) crop their FOV internally, so
    every task is a single invocation; --fast is only meaningful for 'total'.

    NOTE: nnunetv2's preprocessing workers pass torch tensors through /dev/shm.
    Cromwell's GCP Batch backend sizes /dev/shm proportional to VM RAM, so this
    is fine on Terra; a local `docker run` needs --shm-size (the docker default
    of 64 MB crashes with "unable to allocate shared memory")."""
    out_file = work / 'seg.nii.gz'
    cmd = ['TotalSegmentator', '-i', str(nii), '-o', str(out_file), '--ml']
    if ts_task != 'total':
        cmd += ['--task', ts_task]
    if ts_fast:
        if ts_task != 'total':
            raise ValueError(f"fast=True is only supported for task 'total' (got {ts_task!r})")
        cmd.append('--fast')
    print(f'  {" ".join(cmd)}', flush=True)
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0:
        raise RuntimeError(f'TotalSegmentator rc={res.returncode}\n{res.stderr}')
    if not out_file.exists():
        # Defensive: some versions write .nii when handed a .nii.gz suffix or
        # vice versa — accept any NIfTI the call produced in the work dir.
        cands = sorted(work.glob('seg.nii*'))
        if not cands:
            raise RuntimeError('no multilabel NIfTI produced')
        out_file = cands[0]
    return out_file

for uid in series_uids:
    nii = NIFTI_DIR / uid / f'{uid}.nii.gz'
    if not nii.exists():
        cands = list((NIFTI_DIR / uid).glob('*.nii.gz'))
        if not cands:
            errors.append(f'{uid}: no NIfTI found')
            continue
        nii = cands[0]
    print(f'[T+{_elapsed()}] {uid}:', flush=True)
    task_times = {}
    restored_tasks = []
    for t in tasks:
        # fast=True with only the 'total' task selected keeps the old behavior;
        # in a mixed list it applies to 'total' and is ignored for subtasks.
        t_fast = fast and t == 'total'
        if (uid, t) in completed_seg:
            restored_tasks.append(t)
            continue
        work = Path('/tmp/ts_work') / uid
        if work.exists():
            shutil.rmtree(work)
        work.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        try:
            dest = SEG_DIR / uid / t / 'segmentations'
            dest.mkdir(parents=True, exist_ok=True)

            produced = _run_task(nii, work, t, t_fast)
            target = dest / (uid + '.nii.gz')
            if produced.name.endswith('.gz'):
                shutil.move(str(produced), str(target))
            else:
                subprocess.run(f'gzip -c "{produced}" > "{target}"', shell=True, check=True)

            (SEG_DIR / uid / t / 'label_map.json').write_text(
                json.dumps({'model': t, 'labels': TASK_LABELS[t]}, indent=2))
            task_times[t] = round(time.time() - t0, 1)
            print(f'  {t}: {task_times[t]}s')
            if ckpt:
                ckpt.save_seg(SEG_DIR, uid, t)
        except Exception as exc:
            errors.append(f'{uid}/{t}: {traceback.format_exc()}')
            print(f'  ERROR {t}: {exc}')
        finally:
            shutil.rmtree(work, ignore_errors=True)
    # Propagate the exact input NIfTI (identical geometry to the masks) so nb3
    # runs radiomics against it directly. Only when the series dir exists.
    if (SEG_DIR / uid).is_dir():
        shutil.copy(str(nii), str(SEG_DIR / uid / 'reference.nii.gz'))
    usage_metrics['series'][uid] = {'task_s': task_times,
                                    'checkpoint_restored_tasks': restored_tasks}
    if restored_tasks:
        print(f'  restored from checkpoint: {restored_tasks}')

if errors:
    Path('inference_errors.txt').write_text('\n'.join(errors))
print(f'[T+{_elapsed()}] Inference complete ({len(errors)} error(s))')

## Package Boundary-B archive + usage metrics

In [ ]:
import csv
produced = [d for d in SEG_DIR.iterdir() if d.is_dir()]
if not produced:
    raise RuntimeError('No segmentations produced — see inference_errors.txt')

subprocess.run(f'tar -cf - -C {SEG_DIR.parent} {SEG_DIR.name} | lz4 > segmentations.tar.lz4',
               shell=True, check=True)
size_mb = Path('segmentations.tar.lz4').stat().st_size / (1024 ** 2)

usage_metrics['total_elapsed_s'] = round(time.time() - NOTEBOOK_START, 1)
with open('inference_UsageMetrics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['SeriesInstanceUID', 'model', 'model_inference_s', 'run_total_elapsed_s',
                'checkpoint_restored'])
    for uid, m in usage_metrics['series'].items():
        for t, secs in m.get('task_s', {}).items():
            w.writerow([uid, t, secs, usage_metrics['total_elapsed_s'], False])
        for t in m.get('checkpoint_restored_tasks', []):
            w.writerow([uid, t, '', usage_metrics['total_elapsed_s'], True])

# Inference finished and the archive is written: the checkpoint is no longer needed.
if ckpt:
    ckpt.cleanup()

print(f'[T+{_elapsed()}] Wrote segmentations.tar.lz4 ({size_mb:.1f} MB, {len(produced)} series)')